# Our shared Urban + depth model — larger-data near-range training
Notebook revision **PARKING-SHARED-DEPTH-NEAR-2026-10-27-r2-fusion-check**; input/training recipe remains r1. Cross-platform Conv/BN fusion is verified against the exact source and a high-precision reference rather than a Mac-specific fused hash. This is ground-truth training, not distillation. Our shared architecture and input resolution stay unchanged. Only the 57,441-parameter depth decoder trains; Urban weights, detection/corner heads and BatchNorm stay frozen.

**Do not allocate a GPU for staging.** First obtain the CPU-prepared `shared_near_inputs.zip` and its `archive_receipt.json`. Preparation refuses to make an archive unless every vehicle/pedestrian 2–4/4–6 m cell has at least 20 train and 10 dev keyframes, 100 points, and two distinct recording logs per role/cell. These are engineering support minima, not product qualification. Part 1 alone is insufficient. Never move held-out scenes into training to make the gate pass.

**Two matched arms:** broad-depth control and parking-balanced loss. Both start from the previously selected Mini epoch-4 decoder with fresh AdamW state. Both use the same fixed train/dev images, same sampler, seed, 10 epochs per arm, batch 8, LR 0.0003, weight decay 0.0001 and clip norm 5.

**Loss:** raw-metre Huber, delta 0.2. Control = equal-image mean over conservative 2–80 m points. Parking = 80% equal-four-cell near-range loss plus 20% control loss. Four near-range anchors plus four broad draws per batch; sampling can repeat images. An epoch is ceil(train_images/8) updates, not necessarily a visit to every unique image.

Checkpoint selection uses the equal mean of four development-cell per-image MAEs, not broad 2–80 m MAE. No held-out evaluation images are included or opened here. No per-image scale fitting, automatic phone replacement, or metric 3D cuboid claim. Camera-axis sparse depth is not ultrasonic surface range or a 10cm metrology guarantee.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, subprocess, shlex, datetime, hashlib
REVISION = 'PARKING-SHARED-DEPTH-NEAR-2026-10-27-r1'
NOTEBOOK_REVISION = 'PARKING-SHARED-DEPTH-NEAR-2026-10-27-r2-fusion-check'
RUN_ID = 'shared_depth_near_trainval_r1'
MOBILE = Path('/content/mobile_adas3d')
PACKAGE = Path('/content/drive/MyDrive/mobile_adas3d_inputs/parking_shared_near/trainval_inputs_r2')
ARCHIVE = PACKAGE / 'shared_near_inputs.zip'
RECEIPT = PACKAGE / 'archive_receipt.json'
INPUTS = Path('/content/shared_near_inputs_' + RUN_ID)
CACHE = Path('/content/shared_near_features_' + RUN_ID)
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_shared_near') / RUN_ID
SOURCE = Path('/content/Urban_Shared_Near_Source')
WEIGHTS = Path('/content/urban_shared_near_weights')
DEPS = Path('/content/shared_near_task_deps')
ENV = os.environ.copy()
ENV.update(PYTHONPATH=str(DEPS) + os.pathsep + str(MOBILE), OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
READY_DATA = False
GPU_READY = False
def run(command, name):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT / 'colab_logs' / (name + '_' + stamp + '.log')
    log.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    tail = []
    with log.open('w') as stream:
        p = subprocess.Popen(command, cwd=MOBILE if MOBILE.exists() else '/content', env=ENV,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        try:
            for line in p.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush()
                tail = (tail + [line.rstrip()])[-30:]
        finally: p.stdout.close()
        code = p.wait(); stream.write('\nExit code: ' + str(code) + '\n')
    if code: raise RuntimeError(f'Exit {code}; preserve the run and inspect {log}\n' + '\n'.join(tail))
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone',url,MOBILE], 'clone')
branch = subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE,text=True).strip()
if branch != 'main': raise RuntimeError('Use main; preserve existing branches and changes')
run(['git','pull','--ff-only'], 'update')
for name in ['parking_shared_near_data.py','train_parking_shared_near.py','parking_shared_body_verification.py']:
    if not (MOBILE/'scripts'/name).is_file(): raise FileNotFoundError('Required code is not on main: ' + name)
print(NOTEBOOK_REVISION, '— CPU staging first; no old M63/M67 runtime or CUDA extension needed')


## 2. Stage the verified input package — CPU only
Upload both prepared files to PACKAGE above. The package contains original selected RGB images, sparse labels, frozen train/dev roles and the exact warm-start checkpoint. It contains no held-out images.

On the Mac, readiness is checked with `scripts/parking_shared_near_data.py readiness --output <fresh-readiness-folder>`. Once additional audited parts are available, use `prepare --part <part-inputs> <part-coverage> ... --output <fresh-input-folder>`. A failed support gate saves its explanation but creates no training archive. Four staging workers reuse verified local files. All training I/O later uses local storage, not thousands of Drive reads.


In [ ]:
for path in [ARCHIVE, RECEIPT]:
    if not path.is_file(): raise FileNotFoundError(f'CPU package not ready: {path}. Do not activate a GPU yet.')
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''): h.update(block)
    return h.hexdigest()
receipt = json.loads(RECEIPT.read_text())
if receipt['file'] != ARCHIVE.name or digest(ARCHIVE) != receipt['sha256']: raise RuntimeError('Archive/receipt mismatch')
run([sys.executable,MOBILE/'scripts/parking_shared_near_data.py','restore','--archive',ARCHIVE,'--output',INPUTS], 'stage_inputs_cpu')
manifest = json.loads((INPUTS/'input_manifest.json').read_text())
assert manifest['manifest_sha256'] == receipt['manifest_sha256']
assert manifest['revision'] == REVISION and manifest['readiness']['training_data_ready']
assert not manifest['scenes']['test'] and {s['role'] for s in manifest['samples']} == {'train','dev'}
print(json.dumps(manifest['readiness'],indent=2))
READY_DATA = True
print('CPU staging passed. Now select a GPU runtime; after reconnecting rerun sections 1 and 2.')


## 3. GPU preflight and tests
Run only after CPU staging passes. Install the two pinned Urban packages into a task-only directory; do not replace Colab's global PyTorch/CUDA. Runtime versions and GPU identity are recorded. Completed epoch checkpoints resume only with the same run/data/runtime identity; a changed environment or recipe needs a new RUN_ID. An interrupted epoch restarts from the last completed checkpoint.


In [ ]:
if not READY_DATA: raise RuntimeError('Finish CPU staging before selecting/using a GPU')
run([sys.executable,'-m','pip','install','-q','--target',DEPS,'--no-deps','ultralytics==8.4.61','ultralytics-thop==2.0.18'], 'task_dependencies')
run([sys.executable,'-c',
     "import torch,ultralytics; assert torch.cuda.is_available(), 'Select a GPU runtime'; "
     "assert ultralytics.__version__=='8.4.61'; print(torch.__version__,torch.cuda.get_device_name(0))"], 'cuda_preflight')
for pattern in ['test_parking_shared_body_verification.py','test_parking_shared_near.py','test_parking_shared_depth_training.py']:
    run([sys.executable,'-m','unittest','discover','-s','tests','-p',pattern,'-v'], pattern)
SCRIPT = MOBILE/'scripts/train_parking_shared_near.py'
BASE = ['--inputs',INPUTS,'--cache',CACHE,'--source',SOURCE,'--weights',WEIGHTS,'--output',OUTPUT,'--device','cuda']
def stage(name):
    if not GPU_READY: raise RuntimeError('Run GPU preflight first')
    run([sys.executable,'-X','faulthandler','-u',SCRIPT,name,*BASE], name)
GPU_READY = True
print('Ready: two arms, ten epochs per arm, decoder only. Existing phone model unchanged.')


## 4. Cache the frozen image features once
GPU inference only. Both training arms reuse this FP32 local cache. The manifest, runtime, feature-producing functions and frozen Urban state are bound to it; changed inputs cannot silently reuse another run's features. Before caching, every body tensor is checked against high-precision fusion of the exact pinned source; unfused tensors must match exactly. FP32 fusion alone allows atol=rtol=1e-6. The actual runtime body digest remains an exact cache/training guard. The r1 CPU package is unchanged. If the old Mac-hash check stopped before creating run_identity.json, rerun sections 1–4 in the same GPU session; verified local inputs are reused. Never delete or overwrite an existing run identity or feature cache to bypass an error.


In [ ]:
stage('cache')


## 5. Train the two matched arms
Control and parking-balanced arms both restart from the same epoch-4 warm start. Each completed epoch saves optimizer state and checks frozen Urban weights/BN plus an identical detection probe. Training the decoder does not prove detection quality at the new resolution; this probe only proves preservation at a fixed input.

Selection uses only the separate development set's equal-four-cell near-range MAE. This notebook does not access the 128-image evaluation screen or choose a checkpoint from it. No automatic backbone unfreezing, KD, export or iPhone installation occurs.


In [ ]:
stage('train')
result = json.loads((OUTPUT/'near_training_results.json').read_text())
assert result['complete'] and result['frozen_urban_weights_bn_and_detection_probe_unchanged']
assert not result['heldout_evaluation_performed'] and not result['phone_app_changed']
for arm,value in result['arms'].items():
    print(arm, 'selected epoch:',value['selected_epoch'],'dev near-range MAE:',value['selected_dev']['equal_four_cell_mae_m'])
    print(json.dumps(value['selected_dev']['cells'],indent=2))


## 6. Return the result bundle for review
The bundle includes both selected decoder checkpoints, development histories, identities and source snapshots. It excludes images and raw LiDAR. Return it before any further tuning, held-out comparison or phone replacement. A better development score is not a 10cm distance qualification.


In [ ]:
run([sys.executable,SCRIPT,'bundle','--output',OUTPUT], 'bundle')
BUNDLE = OUTPUT/'parking_shared_near_results.zip'
if not BUNDLE.is_file(): raise FileNotFoundError(BUNDLE)
print('Return:',BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
